In [1]:
import os
import tensorflow as tf
from pathlib import Path

print("TENSORFLOW VERSION:", tf.__version__)

print("\nAVAILABLE GPUs:")
print(tf.config.list_physical_devices("GPU"))

data_path = Path(
    "/kaggle/input/datasets/nih-chest-xrays/data"
)

print("\nDATASET AVAILABLE:", data_path.exists())

if data_path.exists():
    print("CSV AVAILABLE:",
          (data_path / "Data_Entry_2017.csv").exists())

    print("IMAGE FOLDERS:",
          len(list(data_path.glob("images_*/images"))))

print("\nENVIRONMENT CHECK COMPLETED!")

TENSORFLOW VERSION: 2.21.0

AVAILABLE GPUs:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

DATASET AVAILABLE: True
CSV AVAILABLE: True
IMAGE FOLDERS: 12

ENVIRONMENT CHECK COMPLETED!


In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Dataset location
data_path = Path(
    "/kaggle/input/datasets/nih-chest-xrays/data"
)

# Load CSV
df_full = pd.read_csv(
    data_path / "Data_Entry_2017.csv"
)

# Find all image files
image_paths = {}

print("SCANNING IMAGE FOLDERS...")

for folder in sorted(data_path.glob("images_*/images")):

    count = 0

    for file in folder.iterdir():

        if file.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            image_paths[file.name] = str(file)
            count += 1

    print(folder.parent.name, ":", count, "images")

# Identify Effusion and No Finding
labels = (
    df_full["Finding Labels"]
    .fillna("")
    .astype(str)
    .str.split("|")
)

effusion_mask = labels.apply(
    lambda x: "Effusion" in x
)

normal_mask = (
    df_full["Finding Labels"].fillna("").str.strip()
    == "No Finding"
)

# Select relevant records
selected_df = df_full[
    effusion_mask | normal_mask
].copy()

selected_df = selected_df.drop_duplicates(
    subset="Image Index"
).copy()

# Assign binary classes
selected_df["Class"] = np.where(
    selected_df["Finding Labels"].str.split("|").apply(
        lambda x: "Effusion" in x
    ),
    "Effusion",
    "No Finding"
)

# Connect image paths
selected_df["Image Path"] = (
    selected_df["Image Index"].map(image_paths)
)

# Verify dataset
print("\nDATASET SUMMARY")
print("Original CSV records:", len(df_full))
print("Selected images:", len(selected_df))
print("Available image files:", len(image_paths))
print("Missing images:", selected_df["Image Path"].isna().sum())

print("\nCLASS DISTRIBUTION:")
print(selected_df["Class"].value_counts())

print("\nUNIQUE PATIENTS:",
      selected_df["Patient ID"].nunique())

# Stop if any selected images are missing
assert selected_df["Image Path"].notna().all(), (
    "Some selected images are missing!"
)

# Save selected dataset information
selected_df.to_csv(
    "/kaggle/working/selected_dataset.csv",
    index=False
)

print("\nDATASET PREPARATION COMPLETED!")

SCANNING IMAGE FOLDERS...
images_001 : 4999 images
images_002 : 10000 images
images_003 : 10000 images
images_004 : 10000 images
images_005 : 10000 images
images_006 : 10000 images
images_007 : 10000 images
images_008 : 10000 images
images_009 : 10000 images
images_010 : 10000 images
images_011 : 10000 images
images_012 : 7121 images

DATASET SUMMARY
Original CSV records: 112120
Selected images: 73678
Available image files: 112120
Missing images: 0

CLASS DISTRIBUTION:
Class
No Finding    60361
Effusion      13317
Name: count, dtype: int64

UNIQUE PATIENTS: 25923

DATASET PREPARATION COMPLETED!


In [3]:
from sklearn.model_selection import StratifiedGroupKFold

# Numerical labels
y = selected_df["Class"].map({
    "No Finding": 0,
    "Effusion": 1
})

groups = selected_df["Patient ID"]

# First split: 90% training/validation, 10% testing
split_test = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

train_val_idx, test_idx = next(
    split_test.split(selected_df, y, groups)
)

train_val_df = selected_df.iloc[train_val_idx].copy()
test_df = selected_df.iloc[test_idx].copy()

# Second split: approximately 80% training, 10% validation
split_val = StratifiedGroupKFold(
    n_splits=9,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(
    split_val.split(
        train_val_df,
        train_val_df["Class"].map({
            "No Finding": 0,
            "Effusion": 1
        }),
        train_val_df["Patient ID"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

# Verify patient separation
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("TRAINING IMAGES:", len(train_df))
print("VALIDATION IMAGES:", len(val_df))
print("TESTING IMAGES:", len(test_df))

print("\nCLASS DISTRIBUTION:")

print(pd.DataFrame({
    "Training": train_df["Class"].value_counts(),
    "Validation": val_df["Class"].value_counts(),
    "Testing": test_df["Class"].value_counts()
}).fillna(0).astype(int))

print("\nPATIENT OVERLAP:")
print("Train/Validation:", len(train_patients & val_patients))
print("Train/Test:", len(train_patients & test_patients))
print("Validation/Test:", len(val_patients & test_patients))

# Save all splits
train_df.to_csv("/kaggle/working/train_split.csv", index=False)
val_df.to_csv("/kaggle/working/validation_split.csv", index=False)
test_df.to_csv("/kaggle/working/test_split.csv", index=False)

print("\nDATASET SPLITTING COMPLETED!")

TRAINING IMAGES: 58428
VALIDATION IMAGES: 7609
TESTING IMAGES: 7641

CLASS DISTRIBUTION:
            Training  Validation  Testing
Class                                    
No Finding     47875        6245     6241
Effusion       10553        1364     1400

PATIENT OVERLAP:
Train/Validation: 0
Train/Test: 0
Validation/Test: 0

DATASET SPLITTING COMPLETED!


In [4]:
import tensorflow as tf
import numpy as np

from sklearn.utils.class_weight import compute_class_weight

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

tf.keras.utils.set_random_seed(42)


# Image preprocessing
def load_and_preprocess(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=1,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32) / 255.0

    image.set_shape([224, 224, 1])

    return image, label


# Create TensorFlow dataset
def make_dataset(dataframe, training=False):

    paths = dataframe["Image Path"].values

    # Effusion = 1, No Finding = 0
    labels = (
        dataframe["Class"] == "Effusion"
    ).astype(np.int32).values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=10000,
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


# Prepare all three datasets
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)


# Calculate class weights
y_train = (
    train_df["Class"] == "Effusion"
).astype(int).values

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weights = {
    0: float(weights[0]),
    1: float(weights[1])
}


# Verify preprocessing
for images, labels in train_ds.take(1):

    print("IMAGE SHAPE:", images.shape)
    print("LABEL SHAPE:", labels.shape)

    print(
        "MINIMUM PIXEL:",
        tf.reduce_min(images).numpy()
    )

    print(
        "MAXIMUM PIXEL:",
        tf.reduce_max(images).numpy()
    )

print("\nCLASS WEIGHTS:")
print(class_weights)

print("\nTRAINING BATCHES:",
      tf.data.experimental.cardinality(train_ds).numpy())

print("\nVALIDATION BATCHES:",
      tf.data.experimental.cardinality(val_ds).numpy())

print("\nPREPROCESSING COMPLETED!")



IMAGE SHAPE: (32, 224, 224, 1)
LABEL SHAPE: (32,)
MINIMUM PIXEL: 0.0
MAXIMUM PIXEL: 1.0

CLASS WEIGHTS:
{0: 0.6102140992167102, 1: 2.7683123282478914}

TRAINING BATCHES: 1826

VALIDATION BATCHES: 238

PREPROCESSING COMPLETED!


In [5]:
import tensorflow as tf

# Image augmentation
augmentation_v4 = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.02),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomTranslation(0.02, 0.02),
    tf.keras.layers.RandomContrast(0.1)
])

# Custom CNN Version 4
model_v4 = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(224, 224, 1)),

    augmentation_v4,

    # Convolution Block 1
    tf.keras.layers.Conv2D(
        32, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Conv2D(
        32, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.MaxPooling2D(2),

    # Convolution Block 2
    tf.keras.layers.Conv2D(
        64, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Conv2D(
        64, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.MaxPooling2D(2),

    # Convolution Block 3
    tf.keras.layers.Conv2D(
        128, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Conv2D(
        128, 3, padding="same", use_bias=False
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.MaxPooling2D(2),

    # Feature extraction
    tf.keras.layers.GlobalAveragePooling2D(),

    # Classification
    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(1, activation="sigmoid")
], name="custom_cnn_v4")


# Compile the model
model_v4.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


# Save architecture configuration
with open(
    "/kaggle/working/cnn_v4_architecture.json", "w"
) as f:
    f.write(model_v4.to_json())


# Display architecture
model_v4.summary()

Model: "custom_cnn_v4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             

 Total params: 304,417 (1.16 MB)

 Trainable params: 303,521 (1.16 MB)

 Non-trainable params: 896 (3.50 KB)

In [6]:
callbacks_v4 = [

    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_cnn_v4.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        "/kaggle/working/training_log_v4.csv"
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir="/kaggle/working/v4_training_backup"
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=0.000001,
        verbose=1
    )
]

print("Training callbacks configured!")
print("Model checkpoint: Enabled")
print("Training history: Enabled")
print("Recovery backup: Enabled")
print("Early stopping: Enabled")

Training callbacks configured!
Model checkpoint: Enabled
Training history: Enabled
Recovery backup: Enabled
Early stopping: Enabled


In [10]:
import numpy as np

# Reuse the existing datasets
train_full_ds = train_ds
val_full_ds = val_ds
test_full_ds = test_ds

# Recalculate balanced class weights
classes, counts = np.unique(y_train, return_counts=True)

class_weights_full = {
    int(cls): len(y_train) / (len(classes) * int(count))
    for cls, count in zip(classes, counts)
}

print("Training dataset:", train_full_ds)
print("Validation dataset:", val_full_ds)
print("Test dataset:", test_full_ds)
print("Class weights:", class_weights_full)
print("All missing variables restored!")z302


Training dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Validation dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Class weights: {0: 0.6102140992167102, 1: 2.7683123282478914}
All missing variables restored!


In [ ]:
history_v4 = model_v4.fit(
    train_full_ds,
    validation_data=val_full_ds,
    epochs=30,
    class_weight=class_weights_full,
    callbacks=callbacks_v4 
)

Epoch 1/30
1826/1826 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - accuracy: 0.6954 - auc: 0.6833 - loss: 0.6083 - pr_auc: 0.3002 - precision: 0.2740 - recall: 0.5504
Epoch 1: val_auc improved from None to 0.78456, saving model to /kaggle/working/best_cnn_v4.keras

Epoch 1: finished saving model to /kaggle/working/best_cnn_v4.keras
1826/1826 ━━━━━━━━━━━━━━━━━━━━ 692s 371ms/step - accuracy: 0.6927 - auc: 0.7482 - loss: 0.5947 - pr_auc: 0.4017 - precision: 0.3301 - recall: 0.6814 - val_accuracy: 0.3807 - val_auc: 0.7846 - val_loss: 0.9715 - val_pr_auc: 0.4041 - val_precision: 0.2197 - val_recall: 0.9619 - learning_rate: 1.0000e-04
Epoch 2/30
1826/1826 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - accuracy: 0.7323 - auc: 0.7839 - loss: 0.5362 - pr_auc: 0.4016 - precision: 0.3387 - recall: 0.6889
Epoch 2: val_auc improved from 0.78456 to 0.79825, saving model to /kaggle/working/best_cnn_v4.keras

Epoch 2: finished saving model to /kaggle/working/best_cnn_v4.keras
1826/1826 ━━━━━━━━━━━━━━━━━━━━ 546s 299ms/st

In [14]:
import pandas as pd

log = pd.read_csv("/kaggle/working/training_log_v4.csv")

print("Total epochs recorded:", len(log))
print("\nBest validation result:")

best_row = log.loc[log["val_auc"].idxmax()]

print("Best epoch:", int(best_row["epoch"]) + 1)
print("Validation accuracy:", round(best_row["val_accuracy"], 4))
print("Validation AUC:", round(best_row["val_auc"], 4))
print("Validation precision:", round(best_row["val_precision"], 4))
print("Validation recall:", round(best_row["val_recall"], 4))

print("\nLast recorded epochs:")
display(log.tail())

Total epochs recorded: 30

Best validation result:
Best epoch: 28
Validation accuracy: 0.8009
Validation AUC: 0.885
Validation precision: 0.4691
Validation recall: 0.8394

Last recorded epochs:


,epoch,accuracy,auc,loss,pr_auc,precision,recall,val_accuracy,val_auc,val_loss,val_pr_auc,val_precision,val_recall
25,25,0.807045,0.894127,0.408181,0.635813,0.480886,0.859471,0.804705,0.884369,0.410656,0.602675,0.474583,0.835044
26,26,0.806514,0.893808,0.409003,0.635076,0.480017,0.855870,0.801551,0.884337,0.415602,0.602392,0.469934,0.836510
27,27,0.807883,0.894195,0.407982,0.634429,0.482097,0.857387,0.800894,0.884963,0.418195,0.604103,0.469070,0.839443
28,28,0.807866,0.894016,0.408161,0.634590,0.482070,0.857292,0.800631,0.884602,0.419364,0.604175,0.468763,0.841642
29,29,0.806137,0.893279,0.410071,0.633807,0.479459,0.855965,0.807333,0.884426,0.404519,0.603614,0.478353,0.826246


In [15]:
import os
import tensorflow as tf

model_path = "/kaggle/working/best_cnn_v4.keras"

print("Checkpoint exists:", os.path.exists(model_path))
print("Checkpoint size:",
      round(os.path.getsize(model_path) / (1024**2), 2), "MB")

best_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

print("\nBest model loaded successfully!")
print("Test dataset available:", "test_full_ds" in globals())
print("Original test dataset available:", "test_ds" in globals())

Checkpoint exists: True
Checkpoint size: 3.59 MB

Best model loaded successfully!
Test dataset available: True
Original test dataset available: True


In [16]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score
)

# Collect predictions and actual labels in the same dataset pass
y_true_list = []
y_prob_list = []

for images, labels_batch in test_full_ds:
    probabilities = best_model(images, training=False).numpy().ravel()

    y_true_list.extend(labels_batch.numpy())
    y_prob_list.extend(probabilities)

y_true = np.array(y_true_list).astype(int)
y_prob = np.array(y_prob_list)

# Standard threshold
y_pred = (y_prob >= 0.5).astype(int)

# Calculate metrics
print("FINAL TEST RESULTS — CNN VERSION 4")
print("-----------------------------------")
print("Total test images:", len(y_true))
print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
print("PR-AUC:", round(average_precision_score(y_true, y_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["No Finding", "Effusion"],
    zero_division=0
))

FINAL TEST RESULTS — CNN VERSION 4
-----------------------------------
Total test images: 7641
Accuracy: 0.8113
Precision: 0.491
Recall: 0.8193
F1-score: 0.614
Balanced accuracy: 0.8144
ROC-AUC: 0.8889
PR-AUC: 0.6283

Confusion Matrix:
[[5052 1189]
 [ 253 1147]]

Classification Report:
              precision    recall  f1-score   support

  No Finding       0.95      0.81      0.88      6241
    Effusion       0.49      0.82      0.61      1400

    accuracy                           0.81      7641
   macro avg       0.72      0.81      0.74      7641
weighted avg       0.87      0.81      0.83      7641



In [17]:

import os
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

# Create a folder for the complete project
backup_dir = Path("/kaggle/working/CNN_V4_Portable_Backup")
backup_dir.mkdir(exist_ok=True)

# ------------------------------------------------
# 1. Copy the trained model and training history
# ------------------------------------------------

model_source = "/kaggle/working/best_cnn_v4.keras"
log_source = "/kaggle/working/training_log_v4.csv"

if not os.path.exists(model_source):
    raise FileNotFoundError("Best CNN model checkpoint is missing!")

shutil.copy2(model_source, backup_dir / "best_cnn_v4.keras")

if os.path.exists(log_source):
    shutil.copy2(log_source, backup_dir / "training_log_v4.csv")

# ------------------------------------------------
# 2. Save dataset splits
# ------------------------------------------------

for name in ["selected_df", "train_df", "val_df", "test_df"]:
    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame):
        obj.to_csv(backup_dir / f"{name}.csv", index=False)
        print("Saved:", name)

# Save the original split indices
if all(name in globals() for name in
       ["train_idx", "val_idx", "test_idx"]):

    np.savez_compressed(
        backup_dir / "split_indices.npz",
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx
    )

# ------------------------------------------------
# 3. Save final test predictions
# ------------------------------------------------

if all(name in globals() for name in
       ["y_true", "y_prob", "y_pred"]):

    np.savez_compressed(
        backup_dir / "test_predictions.npz",
        y_true=np.asarray(y_true),
        y_prob=np.asarray(y_prob),
        y_pred=np.asarray(y_pred)
    )

# ------------------------------------------------
# 4. Save experiment results and configuration
# ------------------------------------------------

experiment = {
    "project": "Pleural Effusion Detection Using CNN",
    "dataset": "NIH Chest X-ray Dataset",
    "task": "Binary image classification",
    "classes": {
        "0": "No Finding",
        "1": "Effusion"
    },
    "input_shape": [224, 224, 1],
    "image_size": "224x224",
    "image_channels": "Grayscale",
    "normalization": "Pixel values divided by 255",
    "augmentation": {
        "rotation": 0.02,
        "zoom": 0.05,
        "translation": [0.02, 0.02],
        "contrast": 0.10
    },
    "model": {
        "architecture": "Custom CNN",
        "convolution_blocks": 3,
        "filters": [32, 64, 128],
        "convolutions_per_block": 2,
        "batch_normalization": True,
        "global_average_pooling": True,
        "dense_units": 128,
        "dropout": 0.4,
        "output_activation": "Sigmoid",
        "parameters": 304417
    },
    "training": {
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "loss": "Binary Crossentropy",
        "batch_size": 32,
        "epochs_completed": 30,
        "best_epoch": 28,
        "class_weights": {
            "0": 0.6102140992167102,
            "1": 2.7683123282478914
        }
    },
    "dataset_counts": {
        "selected_images": 73678,
        "no_finding": 60361,
        "effusion": 13317,
        "training": 58428,
        "validation": 7609,
        "testing": 7641
    },
    "best_validation": {
        "accuracy": 0.8009,
        "roc_auc": 0.8850,
        "precision": 0.4691,
        "recall": 0.8394
    },
    "final_test": {
        "threshold": 0.5,
        "accuracy": 0.8113,
        "precision": 0.4910,
        "recall": 0.8193,
        "f1_score": 0.6140,
        "balanced_accuracy": 0.8144,
        "roc_auc": 0.8889,
        "pr_auc": 0.6283,
        "confusion_matrix": [
            [5052, 1189],
            [253, 1147]
        ]
    },
    "test_images": 7641,
    "patient_level_split": True,
    "decision_threshold": 0.5
}

with open(backup_dir / "experiment_details.json", "w") as f:
    json.dump(experiment, f, indent=4)

# Save the model architecture
if "best_model" in globals():
    with open(backup_dir / "model_architecture.json", "w") as f:
        f.write(best_model.to_json())

    summary_lines = []
    best_model.summary(print_fn=summary_lines.append)

    with open(backup_dir / "model_summary.txt", "w") as f:
        f.write("\n".join(summary_lines))

# Save confusion matrix separately
confusion_df = pd.DataFrame(
    [[5052, 1189], [253, 1147]],
    index=["Actual No Finding", "Actual Effusion"],
    columns=["Predicted No Finding", "Predicted Effusion"]
)
confusion_df.to_csv(backup_dir / "confusion_matrix.csv")

# ------------------------------------------------
# 5. Create a README for the next computer
# ------------------------------------------------

readme = """
CNN VERSION 4 - PORTABLE PROJECT BACKUP

Project:
Pleural Effusion Detection from Chest X-ray Images.

Model:
Custom CNN, binary classification.
Input: 224 x 224 grayscale images.
Classes: No Finding (0), Effusion (1).

IMPORTANT FILES:
best_cnn_v4.keras       - Trained best model checkpoint
training_log_v4.csv    - Training history
experiment_details.json - Model and experiment details
model_architecture.json - Model architecture
model_summary.txt      - Model summary
train_df.csv           - Training split
val_df.csv             - Validation split
test_df.csv            - Test split
selected_df.csv        - Selected dataset metadata
split_indices.npz      - Saved split indices
test_predictions.npz   - Final test predictions
confusion_matrix.csv   - Final test confusion matrix

Final test accuracy: 81.13%
Final test ROC-AUC: 0.8889
Final test F1-score: 0.6140

The original NIH X-ray images are NOT included in this backup.
On another Kaggle computer session, attach the same NIH dataset
as notebook input to recreate the image datasets.

The model was trained without DenseNet or transfer learning.
The final test evaluation used threshold 0.5.

This experiment is for academic research, not clinical diagnosis.
"""

with open(backup_dir / "README.txt", "w") as f:
    f.write(readme)

# ------------------------------------------------
# 6. Package all saved files into one ZIP
# ------------------------------------------------

zip_path = Path("/kaggle/working/CNN_V4_COMPLETE_BACKUP.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in backup_dir.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(backup_dir))

print("\nBACKUP CREATED SUCCESSFULLY!")
print("Location:", zip_path)
print("ZIP size:", round(zip_path.stat().st_size / (1024**2), 2), "MB")

print("\nFILES INCLUDED:")
for file in sorted(backup_dir.iterdir()):
    if file.is_file():
        print(file.name, "-", round(file.stat().st_size / 1024, 1), "KB")


Saved: selected_df
Saved: train_df
Saved: val_df
Saved: test_df



BACKUP CREATED SUCCESSFULLY!
Location: /kaggle/working/CNN_V4_COMPLETE_BACKUP.zip
ZIP size: 4.98 MB

FILES INCLUDED:
README.txt - 1.2 KB
best_cnn_v4.keras - 3676.9 KB
confusion_matrix.csv - 0.1 KB
experiment_details.json - 2.1 KB
model_architecture.json - 18.6 KB
model_summary.txt - 9.2 KB
selected_df.csv - 11429.1 KB
split_indices.npz - 110.6 KB
test_df.csv - 1185.4 KB
test_predictions.npz - 30.6 KB
train_df.csv - 9064.0 KB
training_log_v4.csv - 6.9 KB
val_df.csv - 1180.1 KB
